In [185]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

In [186]:

!curl -O $PREFIX/download.py
!curl -O $PREFIX/embedder.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1376  100  1376    0     0   3717      0 --:--:-- --:--:-- --:--:--  3718
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1520  100  1520    0     0   4497      0 --:--:-- --:--:-- --:--:--  4497


In [187]:
import numpy as np

from embedder import Embedder
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

In [188]:
emb = Embedder()

In [189]:
# from sentence_transformers import SentenceTransformer, util
# model = SentenceTransformer('all-MiniLM-L6-v2')
# query = "How does approximate nearest neighbor search work?"
# v = model.encode(query)
# print(v[0])

In [190]:
query = "How does approximate nearest neighbor search work?"

In [191]:
v = emb.encode(query)

print(v[0])

-0.020582034180917443


In [192]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [f.parse() for f in reader.read()]

In [193]:
doc = next(
    d for d in documents
    if d["filename"] ==
    "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

In [194]:
doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [195]:
doc_vector = emb.encode(doc['content'])
# doc_vector = model.encode(doc['content'])

In [196]:
similarity = doc_vector.dot(v)

print(similarity)

0.36107008472347096


In [197]:
chunks = chunk_documents(
            documents,
            size=2000,
            step=1000
)

In [198]:
vectors = emb.encode_batch(
            [c['content'] for c in chunks]
)

X=np.vstack(vectors)

In [199]:
scores = X.dot(v)

In [200]:
idx=np.argmax(scores)

chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [201]:
query = "What metric do we use to evaluate a search engine?"

q = emb.encode(query)

In [202]:
scores = X.dot(q)

In [203]:
top_indices = np.argsort(scores)[::-1][:5]

results = [chunks[i] for i in top_indices]

In [204]:
for r in results:
    print(r['filename'])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [205]:
print(results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


In [206]:
query = "How do I store vectors in PostgreSQL?"

q = emb.encode(query)

scores = X.dot(q)

top_indices = np.argsort(scores)[::-1][:5]

vector_results = [chunks[i] for i in top_indices]

In [207]:
from minsearch import Index

tindex = Index(
    text_fields=['content']
)

tindex.fit(chunks)

In [208]:
text_results = tindex.search(
    query,
    num_results=5
)

In [209]:
vector_files = set(
    r['filename']
    for r in vector_results
)

text_files = set(
    r['filename']
    for r in text_results
)

print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


In [210]:
def rrf(result_lists, k=60, num_results=5):

    scores = {}
    docs = {}

    for results in result_lists:

        for rank, doc in enumerate(results):

            key = (
                doc['filename'],
                doc['start']
            )

            scores[key] = scores.get(key, 0) + 1/(k+rank)

            docs[key] = doc

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        docs[key]
        for key in ranked[:num_results]
    ]

In [211]:
query = "How do I give the model access to tools?"

q = emb.encode(query)

scores = X.dot(q)

top_indices = np.argsort(scores)[::-1][:5]

vector_results = [chunks[i] for i in top_indices]

In [212]:
text_results = tindex.search(
    query,
    num_results=5
)

In [213]:
results = rrf(
    [vector_results, text_results]
)

In [214]:
print(results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md
